# Demo 01 - device registry, the batch source

The registry says which camera hangs off which recorder, where it is and what it records at. It is
the spine of the whole model: sensor readings arrive with a device id and nothing else, and it is
this file that turns that id into a place someone can walk to.

It arrives as a file, occasionally, exported from another system. That is a batch source, and Auto
Loader handles it the same way it handles a million files: it remembers what it has already read.

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("login", "")
dbutils.widgets.text("target_catalog", "")

login   = dbutils.widgets.get("login")
catalog = dbutils.widgets.get("target_catalog")
assert all([login, catalog])

demo    = f"{login}_demo_bronze"
landing = f"/Volumes/{catalog}/{demo}/demo_landing"
chk     = f"/Volumes/{catalog}/{demo}/checkpoints"

source  = f"{landing}/registry"
target  = f"{catalog}.{demo}.device_registry_bronze"
ckpt    = f"{chk}/registry"

print(source, "->", target)

## Ingest

Three columns get added that the source file does not have:

- `source_file` says which file a row came from, full path so it stays unique
- `ingestion_ts` says when we processed it, which is not the same as when it was produced
- `load_date` is the day partition

Without them a wrong number in the dashboard is impossible to trace back to a file. With them it is
one query.

`_rescued_data` catches anything that does not fit the schema instead of dropping it quietly.

In [0]:
def ingest_registry():
    q = (spark.readStream.format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("header", "true")
         .option("cloudFiles.schemaLocation", f"{ckpt}/schema")
         .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
         .option("rescuedDataColumn", "_rescued_data")
         .load(source)
         .withColumn("source_file",  F.col("_metadata.file_path"))
         .withColumn("ingestion_ts", F.current_timestamp())
         .withColumn("load_date",    F.current_date())
         .writeStream
         .option("checkpointLocation", ckpt)
         .option("mergeSchema", "true")
         .trigger(availableNow=True)
         .toTable(target))
    q.awaitTermination()
    return q


ingest_registry()
print(target, "->", spark.table(target).count(), "rows")

In [0]:
display(spark.table(target)
        .select("camera", "device_id", "location", "zone", "quality",
                "source_file", "ingestion_ts", "load_date")
        .limit(5))

In [0]:
# the registry is what makes a device id mean something
display(spark.table(target)
        .groupBy("device_id")
        .agg(F.count("*").alias("cameras"),
             F.countDistinct("location").alias("locations")))

## Run it again

Nothing in the landing zone changed, so nothing should be loaded. This is the part worth showing
live, because "re-running a load is safe" is easy to claim and takes ten seconds to prove.

It works because the checkpoint keeps a list of files already seen, keyed by full path. The second
run lists the folder, finds only known files, and commits nothing.

In [0]:
before = spark.table(target).count()
ingest_registry()
after = spark.table(target).count()

print(f"{before} -> {after} rows")
assert before == after, "not idempotent"
print("same row count, the second run loaded nothing")

### What actually makes that work

One option: `checkpointLocation`. Under it, in `sources/0/`, Auto Loader keeps a record of the
**full paths** it has already read. The second run lists the folder, finds only known paths, and
produces no micro-batch at all. Nothing is written, so nothing can be duplicated.

On top of that every micro-batch is one atomic Delta commit carrying its batch id, so a run that
dies mid-write resumes without writing the same batch twice.

A real registry update arrives as a new export anyway, `camera_dim_2026-08.csv` and then
`camera_dim_2026-09.csv`. Different paths, so both load, and bronze ends up holding the history of
what the registry looked like over time.

In [0]:
# one commit for the first load, none for the second
display(spark.sql(f"DESCRIBE HISTORY {target}")
        .select("version", "timestamp", "operation",
                F.col("operationMetrics.numOutputRows").alias("rows"))
        .orderBy("version"))